# Analyse de la dépense publique par secteur d'activité (2024–2026)

**Problématique :** La commande publique française bénéficie-t-elle de manière disproportionnée à certains secteurs économiques par rapport à leur poids réel dans le tissu productif ?

### Sources de données mobilisées
1. **Données essentielles de la commande publique (DECP)** : fichiers annuels consolidés (`2024`, `2025`, `2026`) recensant les marchés publics notifiés, leurs montants, durées, procédures et titulaires.
2. **Base Sirene des établissements (`StockEtablissement`, Insee)** : répertoire exhaustif permettant d'identifier le code d'activité principale (`NAF Rev. 2`), la taille salariale, le statut de siège social et la localisation géographique de chaque établissement titulaire.
3. **Données financières des entreprises (INPI / BCE)** : liasses fiscales et ratios financiers conservés pour l'approfondissement financier ultérieur.

In [2]:
# ==============================================================================
# 1. IMPORTS, CONFIGURATION ET CHARGEMENT DES DONNÉES BRUTES (DECP)
# ==============================================================================
import gc
from typing import Optional

import fsspec
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import plotly.express as px
import pyarrow.parquet as pq
import requests
import seaborn as sns

from lire_fichiers import charger_donnees

# Style graphique global
sns.set_theme(style="whitegrid")

# URLs des fichiers Parquet stockés sur MinIO (SSP Cloud)
# Séparer le code (GitHub) des données volumineuses (S3/MinIO) garantit la reproductibilité.
URL_SIRENE_ETAB = (
    "https://minio.lab.sspcloud.fr/glemesre/diffusion/"
    "stock-stocketablissement-parquet.parquet"
)
URL_LIASSES_INPI = (
    "https://minio.lab.sspcloud.fr/k1000ensae/diffusion/projet-2a/data/"
    "donnees_financieres_entreprises.parquet"
)
URL_RATIOS_INPI = (
    "https://minio.lab.sspcloud.fr/glemesre/diffusion/"
    "ratios_inpi_bce(1).parquet"
)

# Paramètres méthodologiques de filtrage des DECP
ANNEE_MIN = 2024          # Fenêtre d'étude retenue (écarte les anciens contrats mis à jour tardivement)
MONTANT_MIN_EUROS = 100   # Écarte les accords-cadres saisis à 1 € symbolique
MONTANT_MAX_EUROS = 5e8   # Écarte les erreurs manifestes de saisie (> 500 M€)

# Chargement des millésimes DECP via le module local `lire_fichiers.py`
donnees_decp = charger_donnees()

for nom_fichier, table in donnees_decp.items():
    print(f"{nom_fichier} : {table.shape[0]:,} lignes, {table.shape[1]} colonnes")

# Aperçu rapide du premier millésime chargé
premier_millesime = next(iter(donnees_decp))
donnees_decp[premier_millesime].head(3)

decp-2024.parquet : 263,037 lignes, 40 colonnes
decp-2025.parquet : 331,165 lignes, 40 colonnes
decp-2026.parquet : 329,312 lignes, 40 colonnes


,id,nature,objet,codeCPV,procedure,dureeMois,dateNotification,datePublicationDonnees,montant,formePrix,...,actesSousTraitance,modificationsActesSousTraitance,modifications,dateSignature,dateDebutExecution,valeurGlobale,montantSubventionPublique,donneesExecution,concessionnaires,autoriteConcedante.id
0,202401L03,Marché,Charpente bois,45214200-2,Procédure adaptée,17,2024-02-08,2003-11-24,49066.80,Unitaire,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024/1111,Marché,LEZTROY LOT1 - FOURNITURE DE REPAS ET GOUTERS ...,55523100-3,Procédure adaptée,48,2024-07-11,2004-05-11,350000.00,Unitaire,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,202402L12,Marché de partenariat,Creation reseaux,45112500-0,Procédure adaptée,14,2024-05-14,2017-02-22,112230.57,Unitaire,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# ==============================================================================
# 2. DIAGNOSTIC DE COMPLÉTUDE (DECP) ET INSPECTION DES SOURCES EXTERNES
# ==============================================================================

import importlib
import src.nettoyage

importlib.reload(src.nettoyage)

from src.nettoyage import analyser_valeurs_manquantes_decp, inspecter_parquet

# --- 2.1. Bilan des valeurs manquantes sur l'ensemble des fichiers DECP ---
bilan_nan, colonnes_vides = analyser_valeurs_manquantes_decp(donnees_decp)
display(bilan_nan.head(15))


# --- 2.2. Fonction utilitaire d'inspection légère des fichiers Parquet distants ---



# Vérification des schémas (Sirene + tables financières INPI réservées pour la suite)
apercu_sirene = inspecter_parquet(URL_SIRENE_ETAB, "Sirene - StockEtablissement")
apercu_ratios = inspecter_parquet(URL_RATIOS_INPI, "Ratios financiers INPI / BCE")

print("\nRépartition des types de bilans non confidentiels (échantillon) :")


del apercu_sirene, apercu_ratios
gc.collect()

Colonnes 100 % vides dans les DECP (0) : []


,nb_nan,pourcentage_nan
donneesExecution,923008,99.95
concessionnaires,922539,99.89
montantSubventionPublique,922539,99.89
dateSignature,922539,99.89
autoriteConcedante.id,922539,99.89
valeurGlobale,922539,99.89
dateDebutExecution,922539,99.89
modificationsActesSousTraitance,893438,96.74
actesSousTraitance,832652,90.16
idAccordCadre,775182,83.94



[Sirene - StockEtablissement] 44,503,711 lignes | 54 colonnes
Colonnes : ['siren', 'nic', 'siret', 'statutDiffusionEtablissement', 'dateCreationEtablissement', 'trancheEffectifsEtablissement', 'anneeEffectifsEtablissement', 'activitePrincipaleRegistreMetiersEtablissement', 'dateDernierTraitementEtablissement', 'etablissementSiege', 'nombrePeriodesEtablissement', 'complementAdresseEtablissement', 'numeroVoieEtablissement', 'indiceRepetitionEtablissement', 'dernierNumeroVoieEtablissement', 'indiceRepetitionDernierNumeroVoieEtablissement', 'typeVoieEtablissement', 'libelleVoieEtablissement', 'codePostalEtablissement', 'libelleCommuneEtablissement', 'libelleCommuneEtrangerEtablissement', 'distributionSpecialeEtablissement', 'codeCommuneEtablissement', 'codeCedexEtablissement', 'libelleCedexEtablissement', 'codePaysEtrangerEtablissement', 'libellePaysEtrangerEtablissement', 'identifiantAdresseEtablissement', 'coordonneeLambertAbscisseEtablissement', 'coordonneeLambertOrdonneeEtablissement'

1156

In [9]:
# ==============================================================================
# 3. CONSOLIDATION ET NETTOYAGE DES DONNÉES DE LA COMMANDE PUBLIQUE (DECP)
# ==============================================================================
importlib.reload(src.nettoyage)

from src.nettoyage import nettoyer_decp

df_decp_raw = pd.concat(donnees_decp.values(), ignore_index=True)

# Suppression des colonnes 100 % vides et standardisation des noms ('.' -> '_')
df_decp_raw = df_decp_raw.drop(columns=colonnes_vides, errors="ignore")
df_decp_raw.columns = df_decp_raw.columns.str.replace(".", "_", regex=False)




df_decp = nettoyer_decp(df_decp_raw)
del df_decp_raw, donnees_decp
gc.collect()
print(f"\nDECP nettoyées prêtes pour la jointure : {len(df_decp):,} lignes")

Contrats antérieurs à 2024 écartés : 140,448
Part de lignes avec SIRET valide (14 chiffres) : 98.8%
Doublons (marché, SIRET) supprimés : 167,144
Lignes écartées pour montant aberrant (<= 100 € ou >= 500,000,000 €) : 3,732

DECP nettoyées prêtes pour la jointure : 712,959 lignes


In [ ]:
# ==============================================================================
# 4. CHARGEMENT SÉLECTIF ET NETTOYAGE DE SIRENE (StockEtablissement)
# ==============================================================================

importlib.reload(src.nettoyage)

from src.nettoyage import charger_et_preparer_sirene

df_sirene, ref_secteurs_sirene = charger_et_preparer_sirene(URL_SIRENE_ETAB)

# --- 1. Aperçu de df_sirene ---
print(f"=== DF_SIRENE : {df_sirene.shape[0]:,} lignes × {df_sirene.shape[1]} colonnes ===")
display(df_sirene.head(10))

print("\nRépartition de l'état administratif (Actifs 'A' vs Fermés 'F') :")
display(df_sirene["etatAdministratifEtablissement"].value_counts(dropna=False).to_frame("nb_etablissements"))

# --- 2. Aperçu complet de ref_secteurs_sirene ---
print(f"\n=== REF_SECTEURS_SIRENE : {len(ref_secteurs_sirene)} sections NAF ===")
ref_affichage = ref_secteurs_sirene.sort_values("nb_etablissements_actifs", ascending=False).copy()
ref_affichage["part_pct"] = (ref_affichage["part_etablissements_sirene"] * 100).round(2)

display(ref_affichage)
print(f"Total établissements actifs : {ref_secteurs_sirene['nb_etablissements_actifs'].sum():,}")
print(f"Somme des parts : {ref_secteurs_sirene['part_etablissements_sirene'].sum():.4f}")

Sirene nettoyé : 44,503,711 établissements uniques prêts pour la jointure


In [ ]:
# ==============================================================================
# 5. JOINTURE SÉCURISÉE (DECP x SIRENE), CONTRÔLE QUALITÉ ET EXPORT
# ==============================================================================

# `how="left"` conserve tous les marchés publics ;
# `validate="many_to_one"` garantit mathématiquement l'absence de duplication de montants.
df_final = pd.merge(
    df_decp,
    df_sirene,
    on="siret",
    how="left",
    validate="many_to_one",
)

# Libération immédiate de la mémoire de la table Sirene complète (~44,5 M de lignes)
del df_decp, df_sirene
gc.collect()

# Diagnostic de couverture de la jointure
taux_lignes = df_final["naf_code"].notna().mean()
montant_tot = df_final["montant_part"].sum()
montant_ok = df_final.loc[df_final["naf_code"].notna(), "montant_part"].sum()

print(f"Table finale (marché x titulaire)      : {len(df_final):,} lignes, {df_final.shape[1]} colonnes")
print(f"Taux d'appariement Sirene (en lignes)  : {taux_lignes:.1%}")
print(f"Taux d'appariement Sirene (en montant) : {montant_ok / montant_tot:.1%}")

# Sauvegarde au format Parquet pour accélérer les sessions suivantes sans recharger Sirene
df_final.to_parquet("decp_sirene_clean.parquet", index=False)
ref_secteurs_sirene.to_parquet("ref_secteurs_sirene.parquet", index=False)

# Tableau de synthèse : poids financier dans les DECP vs poids économique dans Sirene
synthese = (
    df_final.groupby("naf_section", as_index=False)
    .agg(
        nb_contrats=("siret", "count"),
        nb_entreprises=("siren", "nunique"),
        montant_total_M=("montant_part", lambda s: s.sum() / 1e6),
        montant_median=("montant_part", "median"),
    )
    .merge(ref_secteurs_sirene, on="naf_section", how="left")
)

synthese["part_montant_decp"] = (
    synthese["montant_total_M"] / synthese["montant_total_M"].sum()
)
# Indice > 1 : le secteur capte davantage de dépense publique que son poids économique
synthese["indice_surrepresentation"] = (
    synthese["part_montant_decp"] / synthese["part_etablissements_sirene"]
)

display(synthese.sort_values("montant_total_M", ascending=False).head(10))
df_final.head()

### Dictionnaire des variables (`df_final`)

#### 1. Clés de jointure et acteurs
| Variable | Source | Description |
| :--- | :--- | :--- |
| `uid` / `id` | DECP | Identifiant unique du marché public. |
| `siret` | **Clé** | Numéro SIRET normalisé (14 chiffres) de l'établissement titulaire. |
| `siren` | **Créée** | Numéro SIREN (9 chiffres) identifiant l'entreprise mère. |
| `titulaire_id` | DECP | Identifiant brut déclaré pour le titulaire avant nettoyage. |
| `titulaire_type_id` | DECP | Type d'identifiant (filtré uniquement sur `"SIRET"`). |
| `titulaire_nom` | DECP | Raison sociale du titulaire saisie par l'acheteur. |
| `acheteur_id` | DECP | SIRET de l'organisme public acheteur. |
| `acheteur_nom` | DECP | Nom de l'organisme public acheteur (État, collectivité, hôpital…). |

#### 2. Variables financières et temporelles
| Variable | Source | Description |
| :--- | :--- | :--- |
| `montant` | DECP | Montant total brut du contrat en euros (avant partage entre co-titulaires). |
| `nb_titulaires` | **Créée** | Nombre d'entreprises groupées sur le marché (`1` si titulaire unique). |
| `montant_part` | **Créée** | **Variable clé** : part du montant attribuée à l'établissement (`montant / nb_titulaires`). |
| `dureeMois` | DECP | Durée d'exécution prévue du contrat en mois. |
| `montant_annualise` | **Créée** | Montant ramené sur 12 mois (`montant_part / dureeMois * 12`). |
| `dateNotification` | DECP | Date officielle de signature du marché. |

#### 3. Nature et localisation du marché
| Variable | Source | Description |
| :--- | :--- | :--- |
| `nature` | DECP | Type de contrat (`Marché`, `Accord-cadre`, `Concession`…). |
| `procedure` | DECP | Mode de passation (`Appel d'offres ouvert`, `Procédure adaptée`, `Sans publicité`…). |
| `codeCPV` | DECP | Code européen à 8 chiffres décrivant l'objet de l'achat. |
| `cpv_division` | **Créée** | 2 premiers chiffres du CPV (grande famille d'achat, ex. `45` = BTP). |
| `lieuExecution_code` | DECP | Code géographique du lieu d'exécution de la prestation. |
| `lieuExecution_typeCode` | DECP | Échelle géographique (`Département`, `Commune`, `Code postal`, `Région`). |

#### 4. Profil sectoriel et géographique du titulaire (Sirene)
| Variable | Source | Description |
| :--- | :--- | :--- |
| `naf_section` | **Créée** | **Variable sectorielle principale** : les 21 sections NAF (`A` à `U`) avec libellé. |
| `naf_division` | **Créée** | 2 premiers chiffres du code APE (88 divisions fines). |
| `naf_code` | **Créée** | Code APE/NAF nettoyé (ex. `41.20A`). |
| `activitePrincipaleEtablissement` | Sirene | Code APE/NAF brut de l'établissement. |
| `nomenclatureActivitePrincipaleEtablissement` | Sirene | Version de la nomenclature (permet de vérifier `NAFRev2`). |
| `trancheEffectifsEtablissement` | Sirene | Code Insee de la tranche d'effectifs salariés de l'établissement. |
| `etablissementSiege` | Sirene | `True` si le titulaire est le siège social, `False` si c'est une agence locale. |
| `codeCommuneEtablissement` | Sirene | Code commune Insee d'implantation de l'établissement. |
| `dep_titulaire` | **Créée** | Département de l'établissement (`01` à `95`, `2A`/`2B`, `971` à `976`). |
| `etatAdministratifEtablissement` | Sirene | Statut administratif dans Sirene (`A` = actif, `F` = fermé). |

## 6. Visualisations et analyse exploratoire

Les graphiques suivants explorent la répartition sectorielle, la structure des montants par contrat, le rôle des sièges sociaux selon la procédure de passation et la concentration géographique des titulaires.

In [ ]:
# ==============================================================================
# 6.1. VOLUME FINANCIER TOTAL ATTRIBUÉ PAR SECTION NAF (TOUS SECTEURS)
# ==============================================================================

# Sous-ensemble des lignes valides appariées dans Sirene avec montant positif
df_apparies = df_final[
    df_final["naf_code"].notna()
    & (df_final["naf_section"] != "Inconnu")
    & (df_final["montant_part"] > 0)
].copy()

# Agrégation des montants (en M€) et du nombre de marchés distincts par section NAF
totaux_secteurs = (
    df_apparies.groupby("naf_section", observed=True, as_index=False)
    .agg(
        montant_total=("montant_part", "sum"),
        nb_marches=("id", "nunique"),
    )
    .sort_values("montant_total", ascending=False)
)
totaux_secteurs["montant_millions"] = totaux_secteurs["montant_total"] / 1e6

# Graphique en barres horizontales avec hauteur adaptée au nombre de sections
fig_height = max(7, len(totaux_secteurs) * 0.45)
fig, ax = plt.subplots(figsize=(12, fig_height))

sns.barplot(
    data=totaux_secteurs,
    x="montant_millions",
    y="naf_section",
    hue="naf_section",
    palette="viridis",
    legend=False,
    ax=ax,
)

# Ajout des valeurs directement au bout de chaque barre pour lire les petits secteurs
max_val = totaux_secteurs["montant_millions"].max()
for container in ax.containers:
    ax.bar_label(container, fmt=" %.1f M€", padding=4, fontsize=9, color="#222222")

ax.set_xlim(0, max_val * 1.18)
ax.xaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f} M€"))
ax.set_xlabel("Montant total attribué (en millions d'euros)", fontsize=11, labelpad=10)
ax.set_ylabel("")
ax.set_title(
    "Montant total des marchés publics attribués par section NAF de l'établissement",
    fontsize=13,
    fontweight="bold",
    pad=14,
)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 6.2. DISTRIBUTION DES MONTANTS ANNUALISÉS (TOP 10 SECTIONS NAF)
# ==============================================================================

# Sélection des 10 sections les plus fréquentes en nombre de contrats
top_10_sections = df_apparies["naf_section"].value_counts().nlargest(10).index
df_boxplot = df_apparies[
    df_apparies["naf_section"].isin(top_10_sections)
    & (df_apparies["montant_annualise"] > 0)
]

plt.figure(figsize=(11, 6))
sns.boxplot(
    data=df_boxplot,
    x="montant_annualise",
    y="naf_section",
    hue="naf_section",
    order=top_10_sections,
    palette="crest",
    legend=False,
    showfliers=False,  # Masque les valeurs extrêmes pour garder les boîtes interquartiles lisibles
)

plt.xscale("log")
plt.xlabel("Montant annualisé (€, échelle logarithmique)")
plt.ylabel("")
plt.title("Distribution des montants annualisés par grande section NAF (Top 10)")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 6.3. PART DES SIÈGES SOCIAUX SELON LA PROCÉDURE DE PASSATION
# ==============================================================================

# Table de correspondance des codes Insee de tranches d'effectifs salariés
libelles_effectifs = {
    "NN": "0 salarié / Non renseigné",
    "00": "0 salarié",
    "01": "1-2 sal.",
    "02": "3-5 sal.",
    "03": "6-9 sal.",
    "11": "10-19 sal.",
    "12": "20-49 sal.",
    "21": "50-99 sal.",
    "22": "100-199 sal.",
    "31": "200-249 sal.",
    "32": "250-499 sal.",
    "41": "500-999 sal.",
    "42": "1000-1999 sal.",
    "51": "2000-4999 sal.",
    "52": "5000+ sal.",
}
df_apparies["taille_etab"] = df_apparies["trancheEffectifsEtablissement"].map(libelles_effectifs)

# Répartition en volume financier (%) entre agences locales (False) et sièges sociaux (True)
ct_volume = (
    pd.crosstab(
        df_apparies["procedure"],
        df_apparies["etablissementSiege"],
        values=df_apparies["montant_part"],
        aggfunc="sum",
        normalize="index",
    )
    * 100
)

ax = ct_volume.plot(
    kind="barh",
    stacked=True,
    figsize=(10, 5),
    color=["#4c72b0", "#dd8452"],
)
ax.set_xlabel("Part du montant total attribué (%)")
ax.set_ylabel("")
ax.legend(
    ["Agence secondaire (False)", "Siège social (True)"],
    title="Type d'établissement",
    loc="lower right",
)
ax.set_title("Part des montants captés par les sièges sociaux selon la procédure")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 6.4. CARTOGRAPHIE DÉPARTEMENTALE DES MONTANTS ATTRIBUÉS (PLOTLY)
# ==============================================================================

# Chargement du contour simplifié des départements français (GeoJSON)
GEOJSON_URL = (
    "https://raw.githubusercontent.com/gregoiredavid/france-geojson/"
    "master/departements-version-simplifiee.geojson"
)
departements_geo = requests.get(GEOJSON_URL, timeout=30).json()

# Agrégation par département d'implantation de l'établissement titulaire
stats_dep = (
    df_apparies.groupby("dep_titulaire", as_index=False)
    .agg(
        nb_marches=("id", "nunique"),
        montant_total=("montant_part", "sum"),
        montant_median=("montant_part", "median"),
    )
)
stats_dep["montant_millions"] = stats_dep["montant_total"] / 1e6

# Carte choroplèthe interactive
fig = px.choropleth(
    stats_dep,
    geojson=departements_geo,
    locations="dep_titulaire",
    featureidkey="properties.code",
    color="montant_millions",
    hover_name="dep_titulaire",
    hover_data={"nb_marches": True, "montant_millions": ":.1f"},
    color_continuous_scale="Viridis",
    labels={
        "montant_millions": "Montant attribué (M€)",
        "nb_marches": "Nombre de marchés",
    },
    title="Montant des marchés publics par département de l'établissement titulaire",
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()